# 86 Report: the judge that reads results (CPU runtime)

1. **Pick by each method** of notebook 83, now with `judge_v2`'s scores; the method with the highest **validation** EX is kept as `vote_v2`.
2. **Score** every method with the usual evaluation.
3. **Report** both judges side by side. The question it answers: does a judge that **learned** from results do as well as, or better than, the hand-written "rows first" rule of notebook 83?
4. **Final system**: `vote_v1` or `vote_v2`, whichever has the higher validation EX.

**Runtime > Change runtime type > CPU.** About 30 minutes, resumable.

In [ ]:
# ---- effsql setup cell: identical in every notebook (source: notebooks/setup_cell.py) ----
REPO_URL = "https://github.com/selimsidan/correct-is-not-enough"
REPO_DIR = "/content/correct-is-not-enough"
REF = "main"  # branch, tag or commit hash to run
GPU = False  # set True in GPU notebooks: installs the [gen] extra (vLLM)
DRIVE_ROOT = "/content/drive/MyDrive/586"
LOCAL_ROOT = "/content/bird_local"

import os, subprocess, sys
from google.colab import drive

drive.mount("/content/drive")


def sh(*cmd, check=True):
    r = subprocess.run(cmd, text=True, capture_output=True)
    if check and r.returncode != 0:
        raise RuntimeError(f"{' '.join(cmd)} failed:\n{r.stderr}")
    return r.stdout.strip()


if not os.path.isdir(REPO_DIR):
    sh("git", "clone", "--quiet", REPO_URL, REPO_DIR)
sh("git", "-C", REPO_DIR, "fetch", "--quiet", "--all", "--tags")
sh("git", "-C", REPO_DIR, "checkout", "--quiet", REF)
if sh("git", "-C", REPO_DIR, "rev-parse", "--verify", "--quiet", f"origin/{REF}", check=False):
    sh("git", "-C", REPO_DIR, "merge", "--ff-only", "--quiet", f"origin/{REF}")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", REPO_DIR + ("[gen]" if GPU else "")], check=True
)
os.environ["EFFSQL_DRIVE_ROOT"] = DRIVE_ROOT
os.environ["EFFSQL_LOCAL_ROOT"] = LOCAL_ROOT
print("effsql commit:", sh("git", "-C", REPO_DIR, "rev-parse", "HEAD"))

In [ ]:
import json, os
R = f"{DRIVE_ROOT}/effsql/runs"
JUDGE = "judge_v2"        # the judge that reads results (notebook 85)
CAND = "judge_cand_v1"    # candidates, scores, labels and answer-time results
VOTE = "vote_v2"          # answers per method with judge_v2 (vote_v2_<method>) and the kept one (vote_v2)
VOTE1 = "vote_v1"         # the same with judge_v1 (notebook 83)
GEN = "rl_v1_it8"         # the generator
METHODS = ["judge", "judge_compile", "result_vote", "weighted_vote", "result_vote_nonempty", "weighted_vote_nonempty"]
assert json.load(open(f"{R}/{JUDGE}/judge_meta.json"))["complete"], "run 85_judge_results first"
VAL_DBS = ",".join(json.load(open(f"{R}/sft_v1/export_meta.json"))["holdout_db_ids"])

### 1. Pick by each method; keep the best on validation

In [ ]:
!effsql stage --split train --db-ids {VAL_DBS}
!effsql stage --split dev
!python -m effsql.judge vote --run {CAND} --judge {JUDGE} --out {VOTE}

### 2. Score every method with the usual evaluation (resumable)

In [ ]:
MODEL = "Qwen/Qwen2.5-Coder-7B-Instruct"
for m in METHODS:
    r = f"{VOTE}_{m}"
    !python -m effsql.train eval-val --run {r}
    !effsql evaluate --pred {R}/{r}/predictions_dev.jsonl --split dev --timing none --run {r}_dev --model {MODEL} --method {m}
    !effsql evaluate --pred {R}/{r}/predictions_minidev.jsonl --split minidev --timing none --run {r}_minidev --model {MODEL} --method {m}

### 3. Report: both judges, with and without the hand-written rule

Saved as `vote_v2_report/report.txt`.

In [ ]:
RUNS = (["pool_v1_{split}_base", GEN, f"{VOTE1}_judge", f"{VOTE}_judge", f"{VOTE1}_weighted_vote", f"{VOTE}_weighted_vote",
         f"{VOTE1}_weighted_vote_nonempty", f"{VOTE}_weighted_vote_nonempty"]
        + ["ref_omnisql7b", "ref_arctic7b", "ref_qwen32b"])
LABELS = ",".join(["pool_v1_{split}_base=zero-shot 7B", f"{GEN}=generator alone (RL)",
                   f"{VOTE1}_judge=judge v1 (SQL only)", f"{VOTE}_judge=judge v2 (reads results)",
                   f"{VOTE1}_weighted_vote=judge v1 + vote", f"{VOTE}_weighted_vote=judge v2 + vote",
                   f"{VOTE1}_weighted_vote_nonempty=judge v1 + vote + rows first",
                   f"{VOTE}_weighted_vote_nonempty=judge v2 + vote + rows first",
                   "ref_omnisql7b=OmniSQL-7B", "ref_arctic7b=Arctic-R1-7B", "ref_qwen32b=Qwen-32B zero-shot"])
assert all("," not in x.split("=", 1)[1] for x in LABELS.split(",")) and LABELS.count("=") == len(RUNS)
!python -m effsql.train report --runs "{','.join(RUNS)}" --ref {GEN} --labels "{LABELS}" --out vote_v2_report

### 4. Final system: the higher validation EX (ties keep `vote_v1`, chosen first)

In [ ]:
sel = {v: json.load(open(f"{R}/{v}/selection.json")) for v in (VOTE1, VOTE)}
ex = {v: s["splits"]["val"]["methods"][s["chosen"]]["ex"] for v, s in sel.items()}
for v, s in sel.items():
    print(f"{v}: method {s['chosen']}, validation EX {ex[v]:.1f}")
print("final system:", VOTE if ex[VOTE] > ex[VOTE1] else VOTE1)

### Done: disconnect the runtime